In [0]:
from pyspark.sql import functions as F

df_bronze = spark.table("email_triage.bronze.brz_tickets")

In [0]:
df_cleaned = (
    df_bronze
    .withColumn("purchase_date", F.to_date("date_of_purchase", "yyyy-MM-dd"))
    .withColumn("first_response_time", F.to_timestamp("first_response_time", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("time_to_resolution", F.to_timestamp("time_to_resolution", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("customer_age", F.col("customer_age").cast("int"))
    .withColumn("csat_rating", F.col("customer_satisfaction_rating").cast("double"))
    .withColumn("clean_subject", F.trim(F.regexp_replace("ticket_subject", r"\s+", " ")))
    .withColumn("clean_description", F.trim(F.regexp_replace("ticket_description", r"\s+", " ")))
    .filter(F.col("clean_description").isNotNull() & (F.col("clean_description") != ""))
)

df_cleaned.createOrReplaceTempView("tmp_cleaned_tickets")

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW tmp_ai_responses AS
SELECT 
    *,
    ai_query(
        'databricks-gpt-oss-120b',
        CONCAT(
            'Analyze this customer support ticket and return strictly a valid JSON object without any markdown formatting. ',
            'The JSON must have exact keys: "ai_category" (choose from: Billing & Payment, Technical Issue, Order & Logistics, Product Inquiry), ',
            '"ai_sentiment" (choose from: Angry, Frustrated, Neutral), ',
            '"ai_urgency" (choose from: High, Medium, Low), ',
            '"confidence_score" (float between 0.0 and 1.0 indicating your certainty). ',
            'Ticket Subject: ', clean_subject, ' | Description: ', clean_description
        )
    ) AS raw_ai_response
FROM tmp_cleaned_tickets;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW tmp_silver_routed AS
WITH parsed_ai AS (
    SELECT 
        *,
        FROM_JSON(
            raw_ai_response, 
            'ai_category STRING, ai_sentiment STRING, ai_urgency STRING, confidence_score DOUBLE'
        ) AS ai_struct
    FROM tmp_ai_responses
),
flattened_ai AS (
    SELECT 
        *,
        ai_struct.ai_category AS ai_category,
        ai_struct.ai_sentiment AS ai_sentiment,
        ai_struct.ai_urgency AS ai_urgency,
        ai_struct.confidence_score AS confidence_score
    FROM parsed_ai
)
SELECT 
    *,
    CASE 
        WHEN confidence_score < 0.80 THEN 'Manual_Review_Queue'
        WHEN ai_urgency = 'High' AND ai_sentiment = 'Angry' THEN 'Senior_Escalation_Team'
        WHEN ai_category = 'Billing & Payment' THEN 'Finance_Team'
        WHEN ai_category = 'Technical Issue' THEN 'IT_Support_Team'
        WHEN ai_category = 'Order & Logistics' THEN 'Logistics_Team'
        ELSE 'General_CS_Team'
    END AS assigned_pic,
    CASE 
        WHEN confidence_score < 0.80 THEN 'Needs Human Review'
        ELSE 'Auto-Routed by AI'
    END AS routing_status
FROM flattened_ai;

In [0]:
%sql
SELECT 
    *
FROM tmp_silver_routed
LIMIT 10;

In [0]:
df_silver = spark.table("tmp_silver_routed").drop("raw_ai_response", "ai_struct")

target_silver_table = "email_triage.silver.slv_tickets"

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_silver_table)
)